# Aggregate 질문: 행동 구간 수 계산

발표용 예제입니다. 상위 검색 결과 수나 `repeat_count`가 아니라, 전체 행동 이벤트에서 질문과 맞는 구간을 골라 겹침을 정리한 뒤 셉니다. 실제 영상 전체를 사람이 검토한 정답과는 구분합니다.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
from pipeline.search.aggregate_query import parse_count_action, event_matches_action, count_episodes

question = '물을 몇 번 마셨어?'
events = [
    {'id': 1, 'video_id': 'demo', 'start_time': 10, 'end_time': 14, 'action': '물을 마심', 'summary': '', 'repeat_count': 8},
    {'id': 2, 'video_id': 'demo', 'start_time': 12, 'end_time': 15, 'action': '물을 마심', 'summary': '', 'repeat_count': 3},
    {'id': 3, 'video_id': 'demo', 'start_time': 30, 'end_time': 34, 'action': '물을 마심', 'summary': '', 'repeat_count': 5},
    {'id': 4, 'video_id': 'demo', 'start_time': 40, 'end_time': 42, 'action': '물그릇 근처에 있음', 'summary': '', 'repeat_count': 1},
]
action, label = parse_count_action(question)
matched = [event for event in events if event_matches_action(event, action)]
episodes = count_episodes(matched)
print('질문:', question)
print('행동 후보 ID:', [event['id'] for event in matched])
print('정리된 구간:', [(e['start_time'], e['end_time'], e['event_ids']) for e in episodes])
print('확인된 후보 구간 수:', len(episodes))

위 예제에서는 이벤트 1·2가 겹쳐 한 구간이고 이벤트 3은 별도 구간입니다. 이벤트 4는 물그릇 근처에만 있어 제외합니다. `repeat_count`를 더하면 16이지만 계산 결과는 **후보 구간 2개**입니다.

## 실제 데이터로 확인하기

아래 셀은 프로젝트 DB가 연결된 환경에서만 실행합니다. `demo_video_id`를 실제 영상 ID로 바꾸고, 후보 구간을 원본 영상과 대조해 누락·오탐을 기록하세요. 날짜 질문에는 `source_frames`의 촬영 시각을 확인해야 합니다.

In [ ]:
demo_video_id = None  # 예: 'dog_drinking_water'
if demo_video_id:
    from db.behavior_events import get_behavior_events_for_aggregation
    timeline = get_behavior_events_for_aggregation(video_id=demo_video_id)
    matched = [event for event in timeline if event_matches_action(event, 'drink')]
    episodes = count_episodes(matched)
    for episode in episodes:
        print(episode['video_id'], episode['start_time'], episode['end_time'], episode['event_ids'])
    print('영상에서 대조할 후보 구간:', len(episodes))
else:
    print('실제 영상 ID와 DB 연결을 설정하면 실행할 수 있습니다.')